# Example 5 - Sea-Level Fingerprints from GRACE (SlrGRACE)

This notebook computes sea-level fingerprints: the spatially-varying sea-level
response to redistributed surface mass (ice/water), including solid-Earth
deformation, gravitational self-attraction, and rotational feedback (the
"GRD" problem). Surface mass change is taken from a GRACE gravimetry product.
The primary steps include:

1. Build a global spherical mesh and refine it near the coastline.
2. Load a single GRACE epoch and build the model geometry.
3. Parameterize the model (Love numbers, solid-earth settings, timestepping) and
   verify it is a fully-consistent, ready-to-solve model.
4. Solve for a single epoch (requires a compiled ISSM binary + cluster access).
5. Plot the single-epoch sea-level and water-height change.
6. Solve a multi-month transient series.
7. Plot the transient results and the global-mean sea-level (GMSL) time series.

This is a from-scratch pyISSM reimplementation of ISSM's MATLAB `SlrGRACE`
tutorial (https://issm.jpl.nasa.gov/documentation/tutorials/sealevelfingerprints/),
not a line-by-line port — see the pyISSM coding standards for why.

In [ ]:
import numpy as np
import pyissm

---
## Setup your modelling environment

Set the path to the local GRACE data directory. This data is not distributed
with pyISSM — download the product referenced below and point `DATA_DIR` at
it.

In [ ]:
# GRACE Tellus land mass-concentration product (JPL RL05.1 DSTvSCS1411).
# Update this path to point at your own local copy of the data.
DATA_DIR = '/path/to/GRACE_and_supporting_datasets'
GRACE_NC = f'{DATA_DIR}/GRCTellus.JPL.200204_201701.LND.RL05_1.DSTvSCS1411.nc'

---
## 1. Model mesh: global sphere + coastal refinement

Sea-level fingerprints are computed on a global domain, so the mesh is a
sphere rather than a regional planar domain. We build an initial coarse
global mesh with `gmshplanet`, classify each vertex as ocean or land with
`gmtmask`, then refine the mesh so that resolution is finest near the
coastline (where the sea-level response varies most sharply) and coarsest
far from it.

In [ ]:
# Create an empty model and build the initial coarse global mesh.
# gmshplanet takes radius/resolution in km (see its docstring); the
# refinemetric passed to it below is in metres.
RADIUS_KM = 6371.012
md = pyissm.model.Model()
md = pyissm.model.mesh.gmshplanet(md, radius=RADIUS_KM, resolution=150)

print(f'Initial mesh: {md.mesh.numberofvertices} vertices, {md.mesh.numberofelements} elements')

In [ ]:
from pyissm.data.ocean_mask import gmtmask

ocean = gmtmask(md.mesh.lat, md.mesh.long)  # 1 = ocean, 0 = land

### Coastal-distance refinement metric

`gmshplanet`'s `refine`/`refinemetric` arguments drive local mesh resolution
from a per-vertex target size. We derive that target from each vertex's
great-circle distance to the coastline, using a `scipy.spatial.cKDTree`
nearest-neighbour query on 3-D unit vectors (vectorised, not the O(N²) nested
loop the MATLAB tutorial uses).

Since `gmtmask` gives us a per-vertex ocean/land classification rather than
explicit coastline geometry, we approximate "distance to the coast" as the
distance from each vertex to the nearest vertex of the *opposite*
classification — at the current mesh resolution this is a reasonable proxy
for distance to the true coastline, and it is what gets refined in the next
mesh pass anyway. The result is clamped to a floor/ceiling target size:
finer on the ocean side (`mindist_coast`) than the land side (`mindist_land`),
since the sea-level solution is driven by loading over the ocean, and capped
at `maxdist` far from any coast.

In [ ]:
from scipy.spatial import cKDTree


def _coastal_distance_metric(lat, long, ocean, mindist_coast, mindist_land, maxdist, radius=RADIUS_KM * 1e3):

    """
    Build a per-vertex mesh-refinement target size from distance to the coast.

    Parameters
    ----------
    lat : ndarray
        Vertex latitude, decimal degrees.
    long : ndarray
        Vertex longitude, decimal degrees.
    ocean : ndarray
        Per-vertex ocean/land classification (1 = ocean, 0 = land), e.g. from `gmtmask`.
    mindist_coast : float
        Minimum (finest) target size for ocean vertices, metres.
    mindist_land : float
        Minimum (finest) target size for land vertices, metres.
    maxdist : float
        Maximum (coarsest) target size far from the coast, metres.
    radius : float, optional
        Sphere radius, metres. Defaults to the mean Earth radius used elsewhere in this notebook.

    Returns
    -------
    ndarray
        Per-vertex refinement target size, metres, suitable for `gmshplanet`'s `refinemetric`.
    """

    lat_r = np.radians(np.ravel(lat))
    long_r = np.radians(np.ravel(long))
    xyz = np.column_stack([np.cos(lat_r) * np.cos(long_r),
                           np.cos(lat_r) * np.sin(long_r),
                           np.sin(lat_r)])

    is_ocean = np.ravel(ocean).astype(bool)

    # Chord length to the nearest vertex of the opposite class, converted to a
    # great-circle distance via the unit-sphere chord/angle relation.
    chord = np.empty(xyz.shape[0])
    chord[is_ocean], _ = cKDTree(xyz[~is_ocean]).query(xyz[is_ocean])
    chord[~is_ocean], _ = cKDTree(xyz[is_ocean]).query(xyz[~is_ocean])
    dist = 2.0 * np.arcsin(np.clip(chord / 2.0, 0.0, 1.0)) * radius

    metric = np.where(is_ocean,
                      np.clip(dist, mindist_coast, maxdist),
                      np.clip(dist, mindist_land, maxdist))
    return metric

In [ ]:
dist_metric = _coastal_distance_metric(md.mesh.lat, md.mesh.long, ocean,
                                       mindist_coast=150e3, mindist_land=300e3, maxdist=600e3)

# gmshplanet refines relative to a *prior* mesh object, not md.mesh itself -
# md.mesh must be reset to empty before the call (see its docstring).
prior_mesh = md.mesh
md.mesh = pyissm.model.classes.mesh.mesh3dsurface()
md = pyissm.model.mesh.gmshplanet(md, radius=RADIUS_KM, resolution=150,
                                  refine=prior_mesh, refinemetric=dist_metric)

print(f'Refined mesh: {md.mesh.numberofvertices} vertices, {md.mesh.numberofelements} elements')

Re-classify ocean/land on the refined mesh and store it as the model's ocean
level set — negative in the ocean, positive on land, matching the ISSM
level-set sign convention (`ice_levelset`/`ocean_levelset` in
`pyissm.model.classes.mask`).

In [ ]:
ocean = gmtmask(md.mesh.lat, md.mesh.long)
md.mask.ocean_levelset = np.where(ocean == 0, 1.0, -1.0)

### Visual check

`pyissm.plot.plot_mesh2d` grids on the model's projected Cartesian `x`/`y`,
which for this global spherical mesh are 3-D coordinates on the sphere rather
than a flat map — it will not produce a meaningful lat/long view here. A
small notebook-local scatter plot on `md.mesh.lat`/`md.mesh.long` is the
correct choice for this global mesh, not a failure to reuse the existing
plotting helpers (the same applies to the single-epoch and transient plots in
Steps 5 and 7).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(md.mesh.long, md.mesh.lat, c=ocean, cmap='coolwarm_r', s=2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Refined mesh vertices ({md.mesh.numberofvertices}), coloured by ocean (1) / land (0)')
fig.colorbar(sc, ax=ax, label='ocean mask')
plt.show()